# 1. InMemoryCache — 개념 이해용

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini")

message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]

## 1.1. 처음 호출

In [ ]:
%%time
# 첫 번째 호출 - API 실제 호출
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

## 1.2. 캐싱 이후 호출

In [ ]:
%%time
# 두 번째 호출 - 캐시에서 즉시 반환
response = llm.invoke(message)
print(response.content)
# Wall time: 약 1ms (거의 0)

# 2. SQLiteCache — 로컬 영구 저장 (실습 권장)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# 프로젝트 폴더에 .langchain.db 파일로 저장
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

llm = ChatOpenAI(model="gpt-4o-mini")
message = [HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")]

## 2.1. 첫번째 호출

In [ ]:
%%time
# 첫 번째 호출 - API 실제 호출 후 파일에 저장
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

## 2.2. 두번째 호출

In [ ]:
%%time
# 두 번째 호출 - 파일에서 즉시 반환 (프로그램 재시작 후에도 동일)
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~5ms

- 일반 캐시 (정확 일치)   
─────────────────────────────────────────   
"서울에서 맛있는 길거리 음식" → 캐시 적중    
"서울에서 유명한 길거리 음식" → 캐시 미적중 (다른 문자열)    

- 시맨틱 캐시 (의미 유사)    
─────────────────────────────────────────      
"서울에서 맛있는 길거리 음식" → 캐시 저장     
"서울에서 유명한 길거리 음식" → 캐시 적중 (유사도 임계값 초과)    

# 3. RedisSemanticCache 테스트

## 3.1 Redis 접속 구성

In [5]:
import inspect
from langchain_redis import RedisSemanticCache
print(inspect.signature(RedisSemanticCache.__init__))

(self, embeddings: 'Embeddings', redis_url: 'str' = 'redis://localhost:6379', distance_threshold: 'float' = 0.2, ttl: 'Optional[int]' = None, name: 'Optional[str]' = 'llmcache', prefix: 'Optional[str]' = 'llmcache', redis_client: 'Optional[Redis]' = None)


In [ ]:
from dotenv import load_dotenv
import os
import redis

load_dotenv()
REDIS_URL = os.environ['REDIS_URL']

r = redis.from_url(REDIS_URL)
r.flushall()
print("Redis 데이터가 완전히 삭제되었습니다.")

Redis 데이터가 완전히 삭제되었습니다.


In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_redis import RedisSemanticCache, RedisVectorStore                      # 변경
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Redis
# IndexSchema 객체 임포트
from redisvl.schema import IndexSchema
from langchain_redis.config import RedisConfig   # 설정 객체 추가
 
REDIS_URL = os.environ['REDIS_URL']
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. IndexSchema 정의 (Metric: COSINE)
schema_dict = {
    "index": {"name": "cache", "prefix": "cache_prefix"},
    "fields": [
        {"name": "text", "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": 1536,
                "algorithm": "flat",
                "distance_metric": "cosine"
            }
        }
    ]
}
schema = IndexSchema.from_dict(schema_dict)

# 2. RedisConfig 설정
# RedisSemanticCache가 내부적으로 인덱스를 인지하도록 설정을 묶어줍니다.
config = RedisConfig(
    index_name="cache",
    index_schema=schema_dict,
    redis_url=REDIS_URL
)

# 3. RedisSemanticCache 생성 (signature에 맞게 파라미터 수정)
# semantic_cache = RedisSemanticCache(
#     embeddings=embeddings,
#     redis_url=REDIS_URL,
#     distance_threshold=0.1,
#     name="cache",        # signature의 'name'이 인덱스 이름 역할을 합니다.
#     prefix="cache_prefix" # 위에서 설정한 prefix와 동일하게 맞춤
# )

# 3. RedisSemanticCache 생성 (signature에 맞게 파라미터 수정)
semantic_cache = RedisSemanticCache(
    embeddings=embeddings,
    redis_url=REDIS_URL,
    distance_threshold=0.01, # 아주 엄격하게 설정
    name="llmcache",         # 기본값 사용
    prefix="llmcache_prefix"
)

print("시그니처에 맞춘 RedisSemanticCache 설정 완료")


set_llm_cache(semantic_cache)
llm = ChatOpenAI(model="gpt-4o-mini")
print("RedisSemanticCache 연결 완료")

시그니처에 맞춘 RedisSemanticCache 설정 완료
RedisSemanticCache 연결 완료


In [3]:
from dotenv import load_dotenv
import os

from langchain_redis import RedisSemanticCache
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.globals import set_llm_cache

# [중요] 테스트 전 Redis를 완전히 비웁니다. (이게 안 되면 계속 실패합니다)
load_dotenv()
REDIS_URL = os.environ['REDIS_URL']
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# r = redis.from_url(REDIS_URL)
# r.flushall()
# print("1. Redis 클린업 완료")


# [핵심] RedisSemanticCache가 스스로 인덱스를 만들도록 유도하거나, 
# 명확히 일치하는 이름을 넘겨야 합니다.
semantic_cache = RedisSemanticCache(
    embeddings=embeddings,
    redis_url=REDIS_URL,
    distance_threshold=0.1,  # 0.87보다 작으므로 이제 분리되어야 함
    name="llmcache",         # 인덱스 이름 통일
    prefix="llmcache_pfx"    # 프리픽스 통일
)

set_llm_cache(semantic_cache)
llm = ChatOpenAI(model="gpt-4o-mini")
print("2. 캐시 연결 완료 (Metric: 기본값/COSINE 확인 필요)")

# 3. 테스트 실행
from langchain_core.messages import HumanMessage
print("3. 테스트 시작...")

# 첫 번째 질문
print("\n[질문 1] 코스피 설명 중...")
ans1 = llm.invoke([HumanMessage(content="코스피가 뭔지 간단히 설명해줘.")])
print(f"응답 1: {ans1.content[:30]}...")

# 두 번째 질문
print("\n[질문 2] 트럼프 질문 중...")
ans2 = llm.invoke([HumanMessage(content="트럼프의 탄핵 확률은 어느 정도일까")])
print(f"응답 2: {ans2.content[:30]}...")


2. 캐시 연결 완료 (Metric: 기본값/COSINE 확인 필요)
3. 테스트 시작...

[질문 1] 코스피 설명 중...
응답 1: 코스피(KOSPI)는 한국 거래소에서 거래되는 주식의 ...

[질문 2] 트럼프 질문 중...
응답 2: 코스피(KOSPI)는 한국 거래소에서 거래되는 주식의 ...


## 3.2. 첫번째 질문

In [2]:
%%time

from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국거래소에 상장된 대형 주식들의 종합적인 주가 변동을 나타내는 지표입니다.
CPU times: total: 62.5 ms
Wall time: 1.34 s


## 3.3. 동일 질문 반복

In [3]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국거래소에 상장된 대형 주식들의 종합적인 주가 변동을 나타내는 지표입니다.
CPU times: total: 0 ns
Wall time: 127 ms


## 3.4. 유사 질문

In [4]:
%%time
response = llm.invoke([HumanMessage(content="코스피가 뭔지 간단히 설명해줘.")])
print(response.content)

코스피 지수는 한국거래소에 상장된 대형 주식들의 종합적인 주가 변동을 나타내는 지표입니다.
CPU times: total: 15.6 ms
Wall time: 132 ms


## 3.5. 다른 질문

In [5]:
%%time
response = llm.invoke([HumanMessage(content="트럼프의 탄핵 확률은 어느 정도일까")])
print(response.content)

코스피 지수는 한국거래소에 상장된 대형 주식들의 종합적인 주가 변동을 나타내는 지표입니다.
CPU times: total: 0 ns
Wall time: 350 ms


## 3.6. 질문간 거리 확인

In [6]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

# 1. 임베딩 모델 설정 (사용 중인 모델과 동일하게)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. 비교할 두 질의
q1 = "코스피가 뭔지 간단히 설명해줘."
q2 = "트럼프의 탄핵 확률은 어느 정도일까"

# 3. 벡터화
v1 = np.array(embeddings.embed_query(q1))
v2 = np.array(embeddings.embed_query(q2))

# 4. Cosine Distance 계산 (1 - Cosine Similarity)
# 이 값이 distance_threshold와 직접 비교되는 수치입니다.
cosine_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
distance = 1 - cosine_sim

print(f"질문 1: {q1}")
print(f"질문 2: {q2}")
print(f"---")
print(f"계산된 거리(Distance): {distance:.4f}")

# 결과 해석 (0.2 기준)
if distance < 0.2:
    print(f"\n결과: 거리가 {distance:.4f}로 0.2보다 작습니다.")
    print("이 경우 캐시가 '유사한 질문'으로 판단하는 것이 정상입니다. threshold를 더 낮추세요.")
else:
    print(f"\n결과: 거리가 {distance:.4f}로 0.2보다 큽니다.")
    print("물리적 거리는 멀지만 캐시 답변이 같다면, Redis 내에 데이터가 잘못 매칭되어 저장된 상태입니다.")


질문 1: 코스피가 뭔지 간단히 설명해줘.
질문 2: 트럼프의 탄핵 확률은 어느 정도일까
---
계산된 거리(Distance): 0.8711

결과: 거리가 0.8711로 0.2보다 큽니다.
물리적 거리는 멀지만 캐시 답변이 같다면, Redis 내에 데이터가 잘못 매칭되어 저장된 상태입니다.


In [7]:
import numpy as np

q1 = "코스피가 뭔지 간단히 설명해줘."
q2 = "트럼프의 탄핵 확률은 어느 정도일까"

v1 = np.array(embeddings.embed_query(q1))
v2 = np.array(embeddings.embed_query(q2))

# 코사인 거리 계산 (1 - 코사인 유사도)
distance = 1 - (np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
print(f"실제 계산된 거리: {distance:.4f}")


실제 계산된 거리: 0.8710
